In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)

from tensorflow.keras import Model

In [2]:
sentences = [
    "i love transformers",
    "deep learning is amazing",
    "gpt generates text",
    "bert understands language",
    "machine learning is powerful",
    "transformers use attention",
    "artificial intelligence is growing",
    "deep learning uses neural networks"
]


In [3]:
vocab_size = 1000
sequence_length = 20

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(sentences)

tokens = vectorizer(sentences)

print(tokens.shape)

train_x = tokens[:, :-1]
train_y = tokens[:, 1:]

print(train_x.shape)
print(train_y.shape)

(8, 20)
(8, 19)
(8, 19)


In [4]:
class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(
        self,
        sequence_length,
        vocab_size,
        embed_dim
    ):
        super().__init__()

        self.token_embedding = Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )

    def call(self, inputs):

        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embedding(
            inputs
        )

        embedded_positions = self.position_embedding(
            positions
        )

        return (
            embedded_tokens +
            embedded_positions
        )



In [5]:
class GPTBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

    def call(self, inputs):

        attention_output = self.attention(
            inputs,
            inputs,
            use_causal_mask=True
        )

        x = self.norm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(x)

        return self.norm2(
            x + ffn_output
        )



In [6]:
embed_dim = 128
dense_dim = 512
num_heads = 4

inputs = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

x = PositionalEncoding(
    sequence_length,
    vocab_size,
    embed_dim
)(inputs)

for _ in range(4):
    x = GPTBlock(
        embed_dim,
        dense_dim,
        num_heads
    )(x)

outputs = Dense(
    vocab_size,
    activation="softmax"
)(x)

gpt = Model(
    inputs,
    outputs
)

gpt.summary()

gpt.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = gpt.fit(
    train_x,
    train_y,
    batch_size=2,
    epochs=50
)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, None, 128)      │       130,560 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block (GPTBlock)            │ (None, None, 128)      │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_1 (GPTBlock)          │ (None, None, 128)      │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_2 (GPTBlock)          │ (None, None, 128)      │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_3 (GPTBlock)          │ (None, None, 128)      │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, None, 1000)     │       129,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,843,688 (7.03 MB)

 Trainable params: 1,843,688 (7.03 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.6579 - loss: 4.2785   
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8618 - loss: 2.1483
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8618 - loss: 1.6295
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8618 - loss: 1.2386
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8816 - loss: 0.9038
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8882 - loss: 0.7328
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8882 - loss: 0.5978
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9013 - loss: 0.4686
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9013 - loss: 0.3848
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9605 - loss: 0.3294
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9539 - loss: 0.2581
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9803 - loss: 0.20

In [7]:
vocab = vectorizer.get_vocabulary()

index_lookup = dict(
    zip(
        range(len(vocab)),
        vocab
    )
)

seed_text = "i"

generated = seed_text

for i in range(10):

    tokenized_input = vectorizer(
        [generated]
    )[:, :-1]

    predictions = gpt.predict(
        tokenized_input,
        verbose=0
    )

    position = len(
        generated.split()
    ) - 1

    next_token_id = np.argmax(
        predictions[0, position]
    )

    next_word = index_lookup.get(
        next_token_id,
        ""
    )

    if next_word == "":
        break

    generated += " " + next_word

print("Generated Text:")
print(generated)

Generated Text:
i love transformers
